<a href="https://colab.research.google.com/github/Cooljoe67/ML_DSAI/blob/main/7_classification_competition_sample_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training

## Import data and libraries

In [122]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from sklearn import set_config
set_config(transform_output="pandas")

url = "https://drive.google.com/file/d/1L23SxwgqjdUeTKikW-L246yOcI12q0_D/view?usp=drive_link"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
training_data = pd.read_csv(path)

training_data = training_data.set_index('Id')
training_data



In [124]:
X = training_data.drop(columns='Expensive')
y = training_data['Expensive']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include='number').columns
categorical_features = X_train.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (SimpleImputer(strategy='median'), numeric_features),
    (make_pipeline(
        SimpleImputer(strategy='most_frequent'),
        OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ), categorical_features),
)

baseline_model = DummyClassifier(strategy='constant', constant=0)
baseline_model.fit(X_train, y_train)
baseline_model


,strategy,'constant'
,random_state,None
,constant,0


In [ ]:
ordinal_columns = [
    'MSZoning',
    'Condition1',
    'Heating',
    'Foundation',
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'KitchenQual',
    'FireplaceQu',
]

ordinal_categories = [
    ['NA', 'A', 'I', 'C', 'RH', 'RM', 'RL', 'RP', 'FV'],
    ['RRAe', 'RRAn', 'RRNe', 'RRNn', 'Artery', 'Feedr', 'NA', 'Norm', 'PosN', 'PosA'],
    ['NA', 'Grav', 'Wall', 'Floor', 'OthW', 'GasA', 'GasW'],
    ['NA', 'Wood', 'Stone', 'Slab', 'BrkTil', 'CBlock', 'PConc'],
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    ['NA', 'No', 'Mn', 'Av', 'Gd'],
    ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
]

nominal_categorical_columns = [
    column for column in categorical_features
    if column not in ordinal_columns
]

ordinal_preprocessor = make_column_transformer(
    (SimpleImputer(strategy='median'), numeric_features),
    (make_pipeline(
        SimpleImputer(strategy='constant', fill_value='NA'),
        OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        ),
    ), ordinal_columns),
    (make_pipeline(
        SimpleImputer(strategy='constant', fill_value='NA'),
        OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=2, sparse_output=False),
    ), nominal_categorical_columns),
)



In [142]:
tree_numeric_features = X_train.select_dtypes(include='number').columns
tree_categorical_features = X_train.select_dtypes(exclude='number').columns

tree_preprocessor = make_column_transformer(
    (SimpleImputer(strategy='median'), tree_numeric_features),
    (make_pipeline(
        SimpleImputer(strategy='constant', fill_value='NA'),
        OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        ),
    ), ordinal_columns),
    (make_pipeline(
        SimpleImputer(strategy='constant', fill_value='NA'),
        OneHotEncoder(handle_unknown='ignore', sparse_output=False),
    ), [column for column in tree_categorical_features if column not in ordinal_columns]),
)

gridsearch_pipeline = make_pipeline(tree_preprocessor, RandomForestClassifier(random_state=42))

ohe_grid = {
    'columntransformer__pipeline-2__onehotencoder__handle_unknown': ['ignore', 'infrequent_if_exist'],
    'columntransformer__pipeline-2__onehotencoder__min_frequency': [None, 2],
}

param_grid = [
    {
        'randomforestclassifier': [RandomForestClassifier(random_state=42, n_jobs=-1)],
        'randomforestclassifier__n_estimators': [200, 400],
        'randomforestclassifier__max_depth': [None, 12, 20],
        'randomforestclassifier__min_samples_split': [2, 5],
        'randomforestclassifier__min_samples_leaf': [1, 2],
        'randomforestclassifier__class_weight': [None, 'balanced'],
        **ohe_grid,
    },
    {
        'randomforestclassifier': [ExtraTreesClassifier(random_state=42, n_jobs=-1)],
        'randomforestclassifier__n_estimators': [200, 400],
        'randomforestclassifier__max_depth': [None, 12, 20],
        'randomforestclassifier__min_samples_split': [2, 5],
        'randomforestclassifier__min_samples_leaf': [1, 2],
        'randomforestclassifier__class_weight': [None, 'balanced'],
        **ohe_grid,
    },
    {
        'randomforestclassifier': [DecisionTreeClassifier(random_state=42)],
        'randomforestclassifier__max_depth': [None, 6, 12, 18],
        'randomforestclassifier__min_samples_split': [2, 5, 10],
        'randomforestclassifier__min_samples_leaf': [1, 2, 4],
        'randomforestclassifier__class_weight': [None, 'balanced'],
        **ohe_grid,
    },
    ]

search_model = GridSearchCV(
    estimator=gridsearch_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-2,
    refit=True,
    verbose=3
)

search_model.fit(X_train, y_train)

if 'assessment_df' not in globals():
    assessment_df = pd.DataFrame(columns=['train', 'test'], dtype=float)

if 'search_model' not in assessment_df.index:
    assessment_df.loc['search_model', ['train', 'test']] = [pd.NA, pd.NA]

assessment_df.loc['search_model', 'train'] = accuracy_score(
    y_true=y_train,
    y_pred=search_model.predict(X_train),
)
assessment_df.loc['search_model', 'test'] = accuracy_score(
    y_true=y_test,
    y_pred=search_model.predict(X_test),
)

search_model.best_params_

Fitting 5 folds for each of 672 candidates, totalling 3360 fits


/opt/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPi

[CV 2/5] END columntransformer__pipeline-2__onehotencoder__handle_unknown=ignore, columntransformer__pipeline-2__onehotencoder__min_frequency=None, randomforestclassifier=RandomForestClassifier(n_jobs=-1, random_state=42), randomforestclassifier__class_weight=None, randomforestclassifier__max_depth=None, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__min_samples_split=2, randomforestclassifier__n_estimators=200;, score=0.953 total time=   0.3s
[CV 4/5] END columntransformer__pipeline-2__onehotencoder__handle_unknown=ignore, columntransformer__pipeline-2__onehotencoder__min_frequency=None, randomforestclassifier=RandomForestClassifier(n_jobs=-1, random_state=42), randomforestclassifier__class_weight=None, randomforestclassifier__max_depth=None, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__min_samples_split=2, randomforestclassifier__n_estimators=200;, score=0.961 total time=   0.3s
[CV 1/5] END columntransformer__pipeline-2__onehotencoder__han

{'columntransformer__pipeline-2__onehotencoder__handle_unknown': 'ignore',
 'columntransformer__pipeline-2__onehotencoder__min_frequency': None,
 'randomforestclassifier': RandomForestClassifier(n_jobs=-1, random_state=42),
 'randomforestclassifier__class_weight': None,
 'randomforestclassifier__max_depth': 12,
 'randomforestclassifier__min_samples_leaf': 1,
 'randomforestclassifier__min_samples_split': 5,
 'randomforestclassifier__n_estimators': 200}

In [143]:
# test accuracy
test_pred = search_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
print(test_acc)

# other metrics
from sklearn.metrics import recall_score, precision_score, f1_score, cohen_kappa_score
test_recall = recall_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)
test_cohens_kappa= cohen_kappa_score(y_test, test_pred)
print(f"Test Accuracy: {round(test_acc, 3)}\nTest Recall: {round(test_recall, 3)}\nTest Precision: {round(test_precision, 3)}\nTest F1: {round(test_f1, 3)}\nTest Cohen's Kappa: {round(test_cohens_kappa, 3)}")

0.9417808219178082
Test Accuracy: 0.942
Test Recall: 0.674
Test Precision: 0.906
Test F1: 0.773
Test Cohen's Kappa: 0.741


In [144]:
from sklearn.metrics import recall_score, precision_score, f1_score, cohen_kappa_score

baseline_test_pred = baseline_model.predict(X_test)
search_test_pred = search_model.predict(X_test)

comparison_df = pd.DataFrame(
    {
        'baseline_model': [
            accuracy_score(y_test, baseline_test_pred),
            recall_score(y_test, baseline_test_pred),
            precision_score(y_test, baseline_test_pred),
            f1_score(y_test, baseline_test_pred),
            cohen_kappa_score(y_test, baseline_test_pred),
        ],
        'search_model': [
            accuracy_score(y_test, search_test_pred),
            recall_score(y_test, search_test_pred),
            precision_score(y_test, search_test_pred),
            f1_score(y_test, search_test_pred),
            cohen_kappa_score(y_test, search_test_pred),
        ],
    },
    index=['accuracy', 'recall', 'precision', 'f1', 'cohens_kappa'],
)

comparison_df

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,baseline_model,search_model
accuracy,0.85274,0.941781
recall,0.00000,0.674419
precision,0.00000,0.906250
f1,0.00000,0.773333
cohens_kappa,0.00000,0.740756


# Predict on testing data

## Import testing data
Remember, everything done up until now has been performed on your training data. The *real* testing data has no labels (visible to you).

In [145]:
url = "https://drive.google.com/file/d/15PfmTxmavQCT-f7iY9tgwWxm9t4GRees/view?usp=drive_link"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
testing_data = pd.read_csv(path)
testing_data = testing_data.set_index('Id')



Use your model's `.predict()` method to create predictions.

In [146]:
# Use the trained pipeline's predict method. Example:
# preds = search_model.predict(testing_data)
# Or use `ordinal_model`, `final_model`, or another fitted estimator as appropriate.
prediction_features = testing_data.copy()

# Ensure feature set matches training data used by search_model
if 'Id' in X.columns and 'Id' not in prediction_features.columns:
    prediction_features['Id'] = prediction_features.index

prediction_features = prediction_features.reindex(columns=X.columns)
testing_data['Expensive'] = search_model.predict(prediction_features)

The output of `.predict()` is an array. Save this as a column on `testing_data`.

In [147]:
# Export the column 'Expensive' along with the index to create a submission file
testing_data['Expensive'].to_csv('./submission.csv')